In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import time
import re
import matplotlib.pyplot as plt
import io

In [3]:
def get_url(url):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/111.0.0.0 Safari/537.36'}
    r=requests.get(url, headers=headers)
    return r

def get_repo_data_page(page=1):
    url=f'http://www.pbc.gov.cn/zhengcehuobisi/125207/125213/125431/125475/17081/index{page}.html'
    r=get_url(url)
    return r

def get_datalink(page=1):
    r=get_repo_data_page(page)
    soup=BeautifulSoup(r.content, 'html.parser')
    datalink_list=[]
    domain='http://www.pbc.gov.cn'
    for link in soup.find_all('a', {'title': re.compile('交易公告')}):
        datalink_list.append(domain+link.get('href'))
    return datalink_list

def get_repo_data(url):
    try:
        r=get_url(url)
        soup=BeautifulSoup(r.content, 'html.parser')
        table=soup.find_all('div', {'id': 'zoom'})[0]
        if '央行票据' in table.text:
            year=re.findall(r'([0-9]{4}年)', table.text)[0]
            date=year+re.findall(r'((?:[0-9]{4}年)?[0-9]{1,2}月[0-9]{1,2}日).*?央行票据', table.text)[0]
        else:
            date=re.findall(r'((?:[0-9]{4}年)?[0-9]{1,2}月[0-9]{1,2}日).*?逆回购', table.text)[0]
        df=pd.read_html(io.StringIO(str(table)))[0]
        df.columns=df.iloc[0]
        df=df.drop(0)
        df['Date']=pd.to_datetime(date, format='%Y年%m月%d日')
        return df
    except:
        print(url)

def get_links(pages):
    # get all the link from all pages
    datalink_list=[]
    for page in [x+1 for x in range(pages)]:
        datalink_list=datalink_list+get_datalink(page)
    return datalink_list

In [4]:
datalink_list=get_links(5)

In [5]:
df=pd.concat([get_repo_data(url) for url in datalink_list], ignore_index=True)

In [6]:
df.to_clipboard(index=False)

In [7]:
df

,期限,操作利率,投标量,中标量,Date,操作量,期次,发行量 （人民币）,中标利率
0,7天,1.50%,4554亿元,4554亿元,2025-03-26,NaN,NaN,NaN,NaN
1,7天,1.50%,3779亿元,3779亿元,2025-03-25,NaN,NaN,NaN,NaN
2,7天,1.50%,NaN,NaN,2025-03-24,1350亿元,NaN,NaN,NaN
3,7天,1.50%,NaN,NaN,2025-03-21,930亿元,NaN,NaN,NaN
4,7天,1.50%,NaN,NaN,2025-03-20,2685亿元,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
97,7天,1.50%,NaN,NaN,2024-11-12,1255亿元,NaN,NaN,NaN
98,7天,1.50%,NaN,NaN,2024-11-11,1337亿元,NaN,NaN,NaN
99,7天,1.50%,NaN,NaN,2024-11-08,122亿元,NaN,NaN,NaN
100,7天,1.50%,NaN,NaN,2024-11-07,192亿元,NaN,NaN,NaN


In [2]:
'〇'=='○'

False

In [3]:
'〇'=='〇'

True